## Setup

In [1]:
import pandas as pd
import numpy as np
import yaml
import pickle

In [2]:
import sys
sys.path.append('..') 

from shapnarrative_metrics.llm_tools.llm_wrappers import GptApi, LLMWrapper, ClaudeApi, MistralApi,DeepSeekApi
from shapnarrative_metrics.agents.FaithfulEvaluator import FaithfulEvaluator
from shapnarrative_metrics.agents.Narrator import NarratorAgent
from shapnarrative_metrics.agents.FaithfulCritic import FaithfulCritic
from shapnarrative_metrics.agents.Coherence import CoherenceAgent
from shapnarrative_metrics.agents.prompt import Prompt

c:\Users\yhe\Desktop\PHD\Research\25spring_Agentic_AI_project\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### API Key & dataset

In [3]:
with open("../config/keys.yaml") as f:
    dict=yaml.safe_load(f)
api_key = dict["API_keys"]["OpenAI"]
hf_token = dict["API_keys"]["hf_token"]
DeepSeek_API_key = dict["API_keys"]["DeepSeek"]

dataset_name="student"

with open(f'../data/{dataset_name}_dataset/dataset_info', 'rb') as f:
   ds_info= pickle.load(f)

with open(f'../data/{dataset_name}_dataset/RF.pkl', 'rb') as f:
   trained_model=pickle.load(f)

train=pd.read_parquet(f"../data/{dataset_name}_dataset/train_cleaned.parquet")
test=pd.read_parquet(f"../data/{dataset_name}_dataset/test_cleaned.parquet")

print(test.index)

Index([252, 236, 275, 148, 309,  33, 248, 150, 346, 246, 221, 261,   9, 204,
       329, 207, 332,  59, 371,  11, 300, 299, 159, 280, 175, 374, 196, 189,
       188, 211, 307, 284,  36,  72, 114, 274,  82, 345,  54, 385,  26, 190,
       120, 212, 170,  24,  13,  95, 237, 285, 264,  20, 181, 107, 171, 223,
       192,  42,  48,  15, 258, 228,  41,  31,  75, 102, 173, 373, 147,  79,
       293,  91,  52, 179, 200, 199, 320, 375, 245],
      dtype='int64')


#### Choose Data instance & import system message

In [ ]:
idx_list_student=[252, 95, 375, 82, 150, 261, 246, 148, 75, 190, 300, 175, 221, 11, 24, 13, 72, 236, 20, 373]
idx_list_fifa=[4, 40, 96, 19, 84, 69, 80, 105, 45, 31, 11, 118, 26, 56, 81, 55, 94, 113, 104, 10]
idx_list_credit=[769, 94, 361, 980, 804, 809, 124, 966, 675, 175, 571, 990, 700, 319, 929, 241, 938, 146, 83, 882]

In [5]:
selected_indices =[252]

### Four-agent system


In [ ]:
MANIP=False
TEMPERATURE=0.0

# Instantiate LLMs
narrator_llm = DeepSeekApi(DeepSeek_API_key,model="deepseek-chat", system_role="You are a helpful agent that writes model explanations (narratives) based on SHAP values."  , temperature=TEMPERATURE)
faithfulevaluator_llm = DeepSeekApi(DeepSeek_API_key,model="deepseek-chat", system_role="You are an analyst that extracts information from a narrative.", temperature=TEMPERATURE)
faithfulcritic_llm=DeepSeekApi(DeepSeek_API_key,model="deepseek-chat", system_role="You are an agent that summarize the information", temperature=TEMPERATURE)
coherence_llm = DeepSeekApi(DeepSeek_API_key,model="deepseek-chat", system_role="You are a coherence evaluation agent. Your task is to assess the logical coherence of the given narrative explanation. Coherence refers to how well the parts of the narrative fit together to form a unified and logically consistent explanation. ", temperature=TEMPERATURE)


# Initialize Agents
narrator = NarratorAgent(narrator_llm)
faithful_evaluator = FaithfulEvaluator(ds_info=ds_info, llm=faithfulevaluator_llm ) 
# depending on Faithful Critic or Faithful Critic (Rule), initialize it in a different way
faithful_critic = FaithfulCritic(ds_info=ds_info)
# faithful_critic = FaithfulCritic(ds_info=ds_info,llm=faithfulcritic_llm)
coherence_agent = CoherenceAgent(llm=coherence_llm)

# Initialize prompt generator
prompt_generator = Prompt(ds_info=ds_info)

import baseline narrative

In [7]:
import json

baseline_path="../data/baseline_narratives.json"
# baseline_lookup = load_narratives_json()

with open(baseline_path, 'r', encoding='utf-8') as f:
    round0_data = json.load(f)

lookup = {}
for item in round0_data:
    key = (item['dataset'], item['index'])
    lookup[key] = item['round0_narrative']

def get_baseline_narrative(dataset_name, idx):
    """Accesses the preloaded dictionary stored in lookup"""
    return lookup.get((dataset_name, idx), "No Round 0 narrative found.")


Start the agentic system

In [8]:
# Loop through instances
for idx in selected_indices:
    
    print(f"\n🔁 Running for index: {idx}\n" + "-"*40)

    x = test[test.columns[0:-1]].loc[[idx]]
    y = test[test.columns[-1]].loc[[idx]]

    #prepare base prompt
    prompt_generator.gen_variables(trained_model, x, y, tree=True)
    shap_df = pd.DataFrame(prompt_generator.explanation_list[0]) #head(4)
    # print(f"SHAP values for index {idx}:\n{shap_df}\n")
    prompt = prompt_generator.generate_story_prompt(0, prompt_type="long", manipulate=MANIP)
    
    # Round 0 narrative: no-baseline setting
    # narrative = narrator.generate(prompt)
    # print(f"📝 Narrator (Round 0):\n----------------------------\n{narrative}")

    # Round 0 narrative: get baseline narrative
    narrative = get_baseline_narrative(dataset_name,idx)

    print(f"📝 Baseline Narrative Round 0:\n----------------------------\n{narrative}")

    # start the agent system
    for round_num in range(2):
        extraction=faithful_evaluator.generate_extractions([narrative])
        # print(extraction)        
        rank_diff, sign_diff , value_diff, real_rank, extracted_rank=faithful_evaluator.get_diff(extraction[0],shap_df)
        
        df_diff = pd.DataFrame({
            "rank": rank_diff,
            "sign": sign_diff,
            "value": value_diff
        })
        # print(df_diff)
        
        faithful_evaluator_feedback = faithful_evaluator.give_feedback(df_diff, extraction[0])
        print(f"\n🎯 Faithful Evaluator (Feedback {round_num}):\n----------------------------\n{faithful_evaluator_feedback}")

        faithful_feedback = faithful_critic.give_feedback(shap_df,df_diff,extraction[0])
        print(f"\n🎯 Faithful Critic (Feedback {round_num}):\n----------------------------\n{faithful_feedback}")

        coherence_feedback=coherence_agent.give_feedback(narrative)
        print(f"\n💬 Coherence Agent (Feedback {round_num}):\n----------------------------\n{coherence_feedback}")

        # do not stop the iteration even if the feedback indicates "100% faithful" when the Coherence Agent is present
        # if "100% faithful" in faithful_evaluator_feedback.lower():
        #     break

        narrative = narrator.generate(
            initial_prompt=prompt,
            last_narrative=narrative,
            faithful_feedback=faithful_feedback,
            coherence_feedback=coherence_feedback,
        )
        print(f"\n📝 Narrator (Round {round_num+1}):\n----------------------------\n{narrative}")


🔁 Running for index: 252
----------------------------------------
📝 Baseline Narrative Round 0:
----------------------------
The model predicted that this student would not pass (with 49% probability), leaning slightly toward failure. The biggest factor was the student’s past class failures—having one failure (compared to the average of 0.36) significantly reduced their predicted chance of passing, as repeated struggles often indicate ongoing academic challenges. Surprisingly, despite the general trend where students aiming for higher education tend to perform better, this student’s lack of interest in further studies (unlike 95% of their peers) further hurt their odds. Their frequent social outings (rated 5 out of 5, well above average) also played a role, suggesting that less time spent studying may have contributed. Finally, being slightly older than average (18 vs. 16.8) might reflect delays or gaps in their education. Taken together, the student’s academic history, disinterest in

Compute the faithfulness

Test if narratives are identical

In [3]:
from difflib import ndiff

str1 = """

The model predicted with 65% confidence that this customer would repay their loan on time (good credit). The most influential factor was the loan duration—at 12 months, it was significantly shorter than average, strongly increasing the likelihood of repayment since shorter loans are generally less risky. However, the customer’s checking account status (no account) slightly reduced the probability, as lacking an account suggests weaker financial stability. Their younger age (22 years, well below average) also had a small negative impact, possibly due to less established credit history. Lastly, the requested loan amount (1,858 DM) was below average, which modestly improved the prediction, as smaller loans are easier to repay. Overall, the short loan duration and smaller amount outweighed the risks posed by the customer’s youth and lack of a checking account.

"""
str2 = """

The model predicted with 65% confidence that this customer would repay their loan on time (good credit). The most influential factor was the loan duration\u2014at 12 months, it was significantly shorter than average, strongly increasing the likelihood of repayment since shorter loans are generally less risky. However, the customer\u2019s checking account status (no account) slightly reduced the probability, as lacking an account suggests weaker financial stability. Their younger age (22 years, well below average) also had a small negative impact, possibly due to less established credit history. Lastly, the requested loan amount (1,858 DM) was below average, which modestly improved the prediction, as smaller loans are easier to repay. Overall, the short loan duration and smaller amount outweighed the risks posed by the customer\u2019s youth and lack of a checking account.

"""
# Strip trailing/leading spaces before comparison
lines1 = [line.strip() for line in str1.splitlines()]
lines2 = [line.strip() for line in str2.splitlines()]

diff = ndiff(lines1, lines2)

print("🔍 Differences between the two strings:\n")
for line in diff:
    if line.startswith("+ ") or line.startswith("- "):
        print(line)


🔍 Differences between the two strings:

